# Solutions · 06 · Vapour pressure and phase change

Worked solutions to the exercises of [notebook 06](../notebooks/06_vapour_pressure_and_phase_change.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engthermo is missing): install it from GitHub
    import engthermo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engthermo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize
from engthermo import components, eos, vapor
from engthermo.constants import R
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Trouton's rule states that the molar entropy of vaporisation at the normal boiling point is about 85-88 J/(mol K) for most liquids. Test it with the Antoine-implied heats of vaporisation of benzene, toluene, n-hexane, acetone, ethanol and water. Which substances break the rule, and why?

In [2]:
rows = []
for name in ("benzene", "toluene", "n-hexane", "acetone", "ethanol", "water"):
    Tb = vapor.antoine_Tsat(name, 101325.0)
    dS = vapor.antoine_dH_vap(name, Tb) / Tb
    rows.append((name, Tb - 273.15, vapor.antoine_dH_vap(name, Tb) / 1e3, dS))
print(pd.DataFrame(rows, columns=["substance", "Tb (degC)", "dH_vap at Tb (kJ/mol)", "dS_vap (J/(mol K))"]).round(1).to_string(index=False))

substance  Tb (degC)  dH_vap at Tb (kJ/mol)  dS_vap (J/(mol K))
  benzene       80.1                   32.0                90.6
  toluene      110.6                   34.8                90.8
 n-hexane       68.7                   30.6                89.5
  acetone       56.1                   30.9                93.7
  ethanol       78.5                   40.4               114.8
    water      100.0                   41.3               110.6


The hydrocarbons and acetone sit near Trouton's 85-88 J/(mol K): boiling releases a similar amount of
"disorder" for any ordinary liquid. Ethanol and water break the rule with about 110 J/(mol K): hydrogen
bonding orders the liquid, so more entropy is gained on vaporising it. (The Antoine-implied values carry the
ideal-vapour approximation; the true entropies of vaporisation are a few percent lower.)

## Exercise 2

Ammonia is stored as a liquid. Compare the wall pressure for a refrigerated tank at -33 degC with a pressurised tank at 40 degC. Using the enthalpy of vaporisation, estimate the boil-off rate (kg/h) of the refrigerated tank for a heat leak of 5 kW.

In [3]:
for T_C in (-33.3, 40.0):
    T = T_C + 273.15
    print(f"ammonia at {T_C:6.1f} degC: vapour pressure {vapor.antoine_psat('ammonia', T)/1e5:6.2f} bar")
dH = vapor.antoine_dH_vap("ammonia", 239.85)
M = components.get("ammonia").M
print(f"boil-off for a 5 kW heat leak: {5e3 / (dH / M) * 3600:.1f} kg/h  (dH_vap {dH/M/1e3:.0f} kJ/kg)")

ammonia at  -33.3 degC: vapour pressure   1.01 bar
ammonia at   40.0 degC: vapour pressure  15.53 bar
boil-off for a 5 kW heat leak: 12.8 kg/h  (dH_vap 1410 kJ/kg)


The refrigerated tank is at atmospheric pressure - a thin-walled, insulated vessel - but must vent or
re-liquefy about 13 kg/h per 5 kW of heat leak; the pressurised tank needs no refrigeration but walls rated
for 15 bar and more in the sun. Large ammonia storage is refrigerated, small storage pressurised.

## Exercise 3

**Implement it yourself:** fit your own Antoine constants to the vapour pressure of benzene computed with the PR equation between 300 and 500 K (use `scipy.optimize.least_squares` on ln p), and compare with the stored constants.

In [4]:
Ts = np.linspace(300, 500, 40)
p_pr = np.array([vapor.eos_psat("pr", "benzene", T)["p_sat"] for T in Ts])
resid = lambda x: x[0] - x[1] / (Ts + x[2]) - np.log(p_pr)
fit = optimize.least_squares(resid, [21.0, 3000.0, -50.0]).x
stored = components.get("benzene").antoine
print(f"fitted to PR: A = {fit[0]:.3f}, B = {fit[1]:.1f}, C = {fit[2]:.2f};  stored: A = {stored[0]:.3f}, B = {stored[1]:.1f}, C = {stored[2]:.2f}")
print(f"max deviation of the fit from the PR pressures: {np.max(np.abs(np.exp(resid(fit)) - 1)):.2%}")

fitted to PR: A = 21.287, B = 3142.0, C = -31.12;  stored: A = 20.861, B = 2826.3, C = -50.49
max deviation of the fit from the PR pressures: 0.60%


Three Antoine constants reproduce two hundred kelvin of vapour-pressure curve within a fraction of a percent.
The fitted constants differ visibly from the stored ones, yet both sets describe nearly the same curve: the
three constants are strongly correlated (a change in C can be compensated by A and B), so different data
ranges give different sets. Compare Antoine equations by their curves, never by their constants - and always
with their temperature range.